In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib

# pytorch-forecasting / lightning
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from sklearn.metrics import mean_absolute_error, mean_squared_error

c:\Users\user\Documents\imp_1\tftenv\lib\site-packages\pytorch_forecasting\models\base_model.py:24: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
# -----------------------
# CONFIG
# -----------------------
CSV_PATH = "India_upi_like_biller_realistic.csv"   # update if needed
ARTIFACT_DIR = "tft_fixed_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Model config (tweakable)
LOOKBACK_DAYS = 180            # for biller aggregates used by clustering
TEST_DAYS = 90                 # last N days used for validation
N_CLUSTERS = 4
RANDOM_STATE = 42

MAX_ENCODER_LENGTH = 60        # how many past days TFT sees
MAX_PREDICTION_LENGTH = 7      # how many days it predicts
BATCH_SIZE = 64
EPOCHS = 12
LR = 0.03

In [3]:
# -----------------------
# 1. LOAD & BASIC CHECK
# -----------------------
df = pd.read_csv(CSV_PATH, parse_dates=["txn_date"])
df['biller_id'] = df['biller_id'].astype(str)
df = df.sort_values(['biller_id','txn_date']).reset_index(drop=True)

print("Rows:", len(df), "billers:", df['biller_id'].nunique())
print("Date range:", df['txn_date'].min().date(), "->", df['txn_date'].max().date())

Rows: 36500 billers: 100
Date range: 2024-10-01 -> 2025-09-30


In [4]:
# -----------------------
# 2. FESTIVAL / HOLIDAY FLAG (use the festival dates you specified)
# -----------------------
# Using the festival dates you mentioned earlier:
festival_dates = {
    "Dussehra_2024": ["2024-10-12"],
    "Ganpati 2024": ["2024-09-07"],
    "Diwali_2024": ["2024-10-30","2024-10-31"],
    "NewYear_2025": ["2025-01-01"],
    "Holi_2025": ["2025-03-13","2025-03-14"],
    "FYend_2025": ["2025-03-29","2025-03-30","2025-03-31"]
}
# flatten to set
fest_set = set(pd.to_datetime(d) for dates in festival_dates.values() for d in dates)
df['is_festival'] = df['txn_date'].isin(fest_set).astype(int)

In [5]:
# -----------------------
# 3. BILLER-LEVEL AGGREGATES FOR CLUSTERING
#    (we fit GMM on training-period aggregates only to avoid leakage)
# -----------------------
def compute_biller_aggs(df_input, lookback_days=LOOKBACK_DAYS, ref_date=None):
    if ref_date is None:
        ref_date = df_input['txn_date'].max()
    start_date = ref_date - pd.Timedelta(days=lookback_days)
    hist = df_input[(df_input['txn_date'] >= start_date) & (df_input['txn_date'] <= ref_date)].copy()
    ag = hist.groupby('biller_id').agg(
        avg_txn_value_per_day=('txn_value','mean'),
        avg_txn_amount_per_day=('txn_amount','mean'),
        std_txn_value=('txn_value','std'),
        std_txn_amount=('txn_amount','std'),
        days_active=('txn_date','nunique')
    ).fillna(0).reset_index()
    ag['avg_ticket_size'] = (ag['avg_txn_amount_per_day'] / (ag['avg_txn_value_per_day'] + 1e-9)).clip(0,None)
    ag['coeff_var_value'] = ag['std_txn_value'] / (ag['avg_txn_value_per_day'] + 1e-9)
    return ag

In [6]:
# time split: use training split_date so clustering uses only training history
split_date = df['txn_date'].max() - pd.Timedelta(days=TEST_DAYS)
train_df_for_aggs = df[df['txn_date'] <= split_date]
biller_aggs_train = compute_biller_aggs(train_df_for_aggs, lookback_days=LOOKBACK_DAYS)
biller_aggs_all = compute_biller_aggs(df, lookback_days=LOOKBACK_DAYS)  # for later merging

In [7]:
# -----------------------
# 4. GMM CLUSTERING (value vs amount) -> map to quadrant labels
# -----------------------
X_train = biller_aggs_train[['avg_txn_value_per_day','avg_txn_amount_per_day']].values
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)

gmm = GaussianMixture(n_components=N_CLUSTERS, random_state=RANDOM_STATE)
gmm.fit(X_train_s)


GaussianMixture(n_components=4, random_state=42)

In [8]:
# assign clusters to all billers using same scaler + gmm
X_all_s = scaler.transform(biller_aggs_all[['avg_txn_value_per_day','avg_txn_amount_per_day']].values)
cluster_ids = gmm.predict(X_all_s)
cluster_probs = gmm.predict_proba(X_all_s)

biller_aggs_all['gmm_cluster'] = cluster_ids
for i in range(cluster_probs.shape[1]):
    biller_aggs_all[f'gmm_prob_{i}'] = cluster_probs[:,i]

In [9]:

# derive readable quadrant labels by comparing cluster centers to medians
centers = scaler.inverse_transform(gmm.means_)  # centers in original scale
centers_df = pd.DataFrame(centers, columns=['avg_txn_value_per_day','avg_txn_amount_per_day'])
median_value = biller_aggs_all['avg_txn_value_per_day'].median()
median_amount = biller_aggs_all['avg_txn_amount_per_day'].median()

In [10]:
def quadrant_label_from_point(v,a):
    if v >= median_value and a >= median_amount:
        return 'HighValue-HighAmount'
    if v >= median_value and a < median_amount:
        return 'HighValue-LowAmount'
    if v < median_value and a >= median_amount:
        return 'LowValue-HighAmount'
    return 'LowValue-LowAmount'


In [11]:

cluster_label_map = {}
for i,row in centers_df.iterrows():
    cluster_label_map[int(i)] = quadrant_label_from_point(row['avg_txn_value_per_day'], row['avg_txn_amount_per_day'])

biller_aggs_all['cluster_label'] = biller_aggs_all['gmm_cluster'].map(cluster_label_map)

In [12]:
# save clustering artifacts
joblib.dump({'gmm':gmm, 'scaler':scaler, 'cluster_label_map':cluster_label_map}, os.path.join(ARTIFACT_DIR,'gmm_clustering.joblib'))
biller_aggs_all.to_csv(os.path.join(ARTIFACT_DIR,'biller_clusters_summary.csv'), index=False)
print("Clusters assigned and saved. Counts:\n", biller_aggs_all['cluster_label'].value_counts())


Clusters assigned and saved. Counts:
 LowValue-LowAmount      62
HighValue-HighAmount    38
Name: cluster_label, dtype: int64


In [13]:
# -----------------------
# 5. MERGE CLUSTER INFO INTO DAILY DF AND CREATE FEATURES (no leakage)
# -----------------------
df = df.merge(biller_aggs_all[['biller_id','gmm_cluster','cluster_label','avg_ticket_size']], on='biller_id', how='left')

In [14]:
# basic time features
df['time_idx'] = (df['txn_date'] - df['txn_date'].min()).dt.days
df['dow'] = df['txn_date'].dt.dayofweek
df['is_weekend'] = df['dow'].isin([5,6]).astype(int)
df['month'] = df['txn_date'].dt.month
# cyclic encoding for month / day-of-week
df['month_sin'] = np.sin(2*np.pi*(df['month']/12))
df['month_cos'] = np.cos(2*np.pi*(df['month']/12))
df['dow_sin'] = np.sin(2*np.pi*(df['dow']/7))
df['dow_cos'] = np.cos(2*np.pi*(df['dow']/7))

In [15]:
# festival flag already computed above? if not, compute again defensively:
df['is_festival'] = df['txn_date'].isin(fest_set).astype(int)

In [16]:
# rolling & lag features (computed per biller) -> always shift so no leakage
lag_days = [1,7,14]
for lag in lag_days:
    df[f'txn_value_lag_{lag}'] = df.groupby('biller_id')['txn_value'].shift(lag)
    df[f'txn_amount_lag_{lag}'] = df.groupby('biller_id')['txn_amount'].shift(lag)


In [17]:
# rolling means and volatility (shifted by 1 to exclude current day)
for w in [7,30]:
    df[f'txn_value_roll_mean_{w}'] = df.groupby('biller_id')['txn_value'].shift(1).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f'txn_value_roll_std_{w}'] = df.groupby('biller_id')['txn_value'].shift(1).rolling(w, min_periods=1).std().reset_index(level=0, drop=True).fillna(0)
    df[f'txn_amount_roll_mean_{w}'] = df.groupby('biller_id')['txn_amount'].shift(1).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f'txn_amount_roll_std_{w}'] = df.groupby('biller_id')['txn_amount'].shift(1).rolling(w, min_periods=1).std().reset_index(level=0, drop=True).fillna(0)


In [18]:
# cluster probability features (soft assignments)
prob_cols = [c for c in biller_aggs_all.columns if c.startswith('gmm_prob_')]
df = df.merge(biller_aggs_all[['biller_id'] + prob_cols], on='biller_id', how='left')

In [19]:
# fill small amount of NA after shifts (encoder will drop rows without enough history later)
# do not fill target columns
feature_fill_cols = ['avg_ticket_size','txn_value_lag_1','txn_value_lag_7','txn_value_roll_mean_7','txn_value_roll_mean_30'] + prob_cols
df[feature_fill_cols] = df[feature_fill_cols].fillna(0)

In [20]:
# -----------------------
# 6. PREPARE TimeSeriesDataSet for TFT
#    - use GroupNormalizer(transformation='softplus') because targets are positive & skewed
#    - include static, known, unknown features carefully
# -----------------------
# Filter out billers with extremely short history (optional)
min_history_days = MAX_ENCODER_LENGTH + 1
valid_billers = df.groupby('biller_id')['txn_date'].nunique()
valid_billers = valid_billers[valid_billers >= min_history_days].index
df = df[df['biller_id'].isin(valid_billers)].copy()
print("After filtering billers with >= {} days history: {} billers remain".format(min_history_days, df['biller_id'].nunique()))

After filtering billers with >= 61 days history: 100 billers remain


In [21]:

# cutoff for training
training_cutoff = df['time_idx'].max() - MAX_PREDICTION_LENGTH - 30  # leave some val buffer

In [22]:
# define which features are static vs known vs unknown
static_categoricals = ['gmm_cluster','cluster_label']
static_reals = ['avg_ticket_size']

In [23]:
time_varying_known_reals = ['time_idx','month','month_sin','month_cos','dow','dow_sin','dow_cos','is_weekend','is_festival'] + prob_cols
time_varying_unknown_reals = ['txn_value','txn_value_lag_1','txn_value_lag_7','txn_value_roll_mean_7','txn_value_roll_mean_30','txn_value_roll_std_7','txn_value_roll_std_30']


In [24]:
df = df.sort_values(["biller_id", "time_idx"])


In [25]:
lag_roll_cols = [
    'txn_value_lag_1','txn_value_lag_7',
    'txn_value_roll_mean_7','txn_value_roll_mean_30',
    'txn_value_roll_std_7','txn_value_roll_std_30'
]

df[lag_roll_cols] = df.groupby("biller_id")[lag_roll_cols].apply(
    lambda g: g.fillna(method='ffill').fillna(method='bfill')
)
df[lag_roll_cols] = df[lag_roll_cols].fillna(0)


C:\Users\user\AppData\Local\Temp\ipykernel_2348\397130784.py:7: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  df[lag_roll_cols] = df.groupby("biller_id")[lag_roll_cols].apply(


In [26]:
print(df[lag_roll_cols].isna().sum())


txn_value_lag_1           0
txn_value_lag_7           0
txn_value_roll_mean_7     0
txn_value_roll_mean_30    0
txn_value_roll_std_7      0
txn_value_roll_std_30     0
dtype: int64


In [27]:
# build dataset
df["gmm_cluster"] = df["gmm_cluster"].astype(str)

training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='txn_value',
    group_ids=['biller_id'],
    min_encoder_length=30,                 # allow variable encoder length >= 30 (you can reduce)
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=static_categoricals,
    static_reals=static_reals,
    time_varying_known_categoricals=[],
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=GroupNormalizer(groups=["biller_id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

In [28]:
# validation dataset (entire df used for prediction)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

In [29]:
# dataloaders
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

In [30]:
print("Training samples:", len(training))
print("Validation samples:", len(validation))

Training samples: 33400
Validation samples: 100


In [31]:
from pytorch_lightning.loggers.logger import Logger

class NoOpExperiment:
    def add_figure(self, *args, **kwargs):
        pass
    def add_image(self, *args, **kwargs):
        pass
    def add_scalar(self, *args, **kwargs):
        pass
    def add_histogram(self, *args, **kwargs):
        pass
    def add_text(self, *args, **kwargs):
        pass


class NoOpLogger(Logger):
    @property
    def name(self):
        return "noop"

    @property
    def version(self):
        return "0"

    @property
    def experiment(self):
        return NoOpExperiment()

    def log_metrics(self, metrics, step=None):
        pass

    def log_hyperparams(self, params):
        pass


In [32]:
# -----------------------
# 7. TRAIN TFT
# -----------------------

logger = NoOpLogger() #added later

pl.seed_everything(RANDOM_STATE)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LR,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    output_size=7,            # 7 quantiles by default
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    enable_checkpointing=False,
    default_root_dir=ARTIFACT_DIR,
    log_every_n_steps=1,
    logger=logger,
    enable_model_summary=False,
    enable_progress_bar=True,
)






Global seed set to 42
c:\Users\user\Documents\imp_1\tftenv\lib\site-packages\pytorch_lightning\utilities\parsing.py:269: UserWarning: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
  rank_zero_warn(
c:\Users\user\Documents\imp_1\tftenv\lib\site-packages\pytorch_lightning\utilities\parsing.py:269: UserWarning: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
  rank_zero_warn(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [33]:
# =========================================
# Disable ALL logging inside the TFT model
# =========================================
tft.logging_metrics = None         # disable metric modules
tft.log = lambda *a, **k: None     # disable lightning log()
tft.log_dict = lambda *a, **k: None
tft.log_metrics = lambda *a, **k: None
if hasattr(tft, "log_image"):
    tft.log_image = lambda *a, **k: None
if hasattr(tft, "log_graph"):
    tft.log_graph = lambda *a, **k: None


In [34]:
tft.validation_step_outputs = []   # disable storage of validation outputs
tft.test_step_outputs = []


In [35]:
# Put this block immediately before trainer.fit(...)
import matplotlib
# Force non-interactive backend (no GUI)
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pytorch_forecasting
import types
from pytorch_lightning.loggers.logger import Logger

# 1) Defensive matplotlib wrappers: close figures immediately to avoid memory + plotting
plt.rcParams["figure.max_open_warning"] = 10000

_orig_subplots = plt.subplots
def _safe_subplots(*args, **kwargs):
    fig, ax = _orig_subplots(*args, **kwargs)
    # make sure fig is closed as soon as possible if someone calls plt.show()
    return fig, ax
plt.subplots = _safe_subplots

# Force plt.show / plt.pause to not hang or create displays; close any created figures
def _safe_show(*args, **kwargs):
    try:
        plt.close("all")
    except Exception:
        pass
plt.show = _safe_show

def _safe_pause(*args, **kwargs):
    return None
plt.pause = _safe_pause

# 2) Disable common PF plotting routines (no-op)
try:
    pytorch_forecasting.metrics.QuantileLoss.plot = lambda *a, **k: None
except Exception:
    pass
# there may be other plotting functions — wrap generically if needed
for attr in ["plot", "plot_prediction", "plot_prediction_examples"]:
    try:
        if hasattr(pytorch_forecasting.metrics, attr):
            setattr(getattr(pytorch_forecasting.metrics, attr), "plot", lambda *a, **k: None)
    except Exception:
        pass

# 3) Define a strict No-Op logger to satisfy Lightning/PF calls to logger.experiment.add_figure(...)
class NoOpExperiment:
    def add_figure(self, *args, **kwargs): pass
    def add_image(self, *args, **kwargs): pass
    def add_scalar(self, *args, **kwargs): pass
    def add_histogram(self, *args, **kwargs): pass
    def add_text(self, *args, **kwargs): pass

class NoOpLogger(Logger):
    @property
    def name(self): return "noop"
    @property
    def version(self): return "0"
    @property
    def experiment(self): return NoOpExperiment()
    def log_hyperparams(self, params): pass
    def log_metrics(self, metrics, step=None): pass

# 4) Ensure trainer has a logger and no callbacks that try to plot.
# If you already created trainer, re-create it with this logger and no callbacks:
logger = NoOpLogger()
trainer_kwargs = dict(
    max_epochs=EPOCHS,
    accelerator="auto",
    logger=logger,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=True,
    log_every_n_steps=1,     # must not be 0
    callbacks=[]             # empty to avoid PF plotting callbacks
)

# If you already built `trainer` earlier, re-create:
import pytorch_lightning as pl
trainer = pl.Trainer(**trainer_kwargs)

# 5) Final safety: ensure PF plotting callbacks are disabled on the model (if any)
try:
    # remove any callbacks added to the model object
    if hasattr(tft, "logging_metrics") and tft.logging_metrics is not None:
        tft.logging_metrics = None
    if hasattr(tft, "loss") and tft.loss is not None:
        pass  # no-op; keep loss
except Exception:
    pass


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [36]:
# Put this block IMMEDIATELY before trainer.fit(...)
import matplotlib
matplotlib.use("Agg")  # non-interactive backend

import matplotlib.pyplot as plt
import numpy as np
import pytorch_forecasting

# ---- 1) Defensive: override pyplot.errorbar to clamp yerr >= 0 ----
_orig_py_errorbar = plt.errorbar

def _safe_py_errorbar(*args, **kwargs):
    # If yerr provided, clamp negative values to zero
    if "yerr" in kwargs and kwargs["yerr"] is not None:
        yerr = np.array(kwargs["yerr"], copy=True)
        # handle shapes: (2, N) or (N,) or (N,2) etc. Use absolute clamp
        yerr = np.where(np.isnan(yerr), 0.0, yerr)
        yerr = np.clip(yerr, a_min=0.0, a_max=None)
        kwargs["yerr"] = yerr
    elif len(args) >= 3:
        # signature: plt.errorbar(x, y, yerr, ...)
        a = list(args)
        if a[2] is not None:
            try:
                yerr = np.array(a[2], copy=True)
                yerr = np.where(np.isnan(yerr), 0.0, yerr)
                yerr = np.clip(yerr, a_min=0.0, a_max=None)
                a[2] = yerr
                args = tuple(a)
            except Exception:
                pass
    return _orig_py_errorbar(*args, **kwargs)

plt.errorbar = _safe_py_errorbar

# ---- 2) Defensive: override Axes.errorbar as well (covers many callers) ----
_orig_axes_errorbar = matplotlib.axes.Axes.errorbar

def _safe_axes_errorbar(self, *args, **kwargs):
    if "yerr" in kwargs and kwargs["yerr"] is not None:
        yerr = np.array(kwargs["yerr"], copy=True)
        yerr = np.where(np.isnan(yerr), 0.0, yerr)
        yerr = np.clip(yerr, a_min=0.0, a_max=None)
        kwargs["yerr"] = yerr
    elif len(args) >= 3:
        a = list(args)
        try:
            yerr = np.array(a[2], copy=True)
            yerr = np.where(np.isnan(yerr), 0.0, yerr)
            yerr = np.clip(yerr, a_min=0.0, a_max=None)
            a[2] = yerr
            args = tuple(a)
        except Exception:
            pass
    return _orig_axes_errorbar(self, *args, **kwargs)

matplotlib.axes.Axes.errorbar = _safe_axes_errorbar

# ---- 3) Disable PF plotting helpers (extra safety) ----
def _no_plot(*a, **k): return None

# common PF plotting hooks
for attr in ["QuantileLoss", "MAE", "RMSE", "MAPE"]:
    try:
        metric = getattr(pytorch_forecasting.metrics, attr, None)
        if metric is not None and hasattr(metric, "plot"):
            setattr(metric, "plot", _no_plot)
    except Exception:
        pass

# also disable any plotting helper methods possibly present
try:
    if hasattr(pytorch_forecasting, "plotting"):
        for name in dir(pytorch_forecasting.plotting):
            obj = getattr(pytorch_forecasting.plotting, name)
            if callable(obj):
                try:
                    setattr(pytorch_forecasting.plotting, name, lambda *a, **k: None)
                except Exception:
                    pass
except Exception:
    pass

# ---- 4) Ensure model won't try to log images or figures ----
# do this AFTER you create `tft`
try:
    tft.logging_metrics = None
    tft.log = lambda *a, **k: None
    tft.log_dict = lambda *a, **k: None
    tft.log_metrics = lambda *a, **k: None
    if hasattr(tft, "log_image"):
        tft.log_image = lambda *a, **k: None
except Exception:
    pass

# ---- final note ----
print("Defensive plotting monkeypatches installed (errorbar clamped).")



Defensive plotting monkeypatches installed (errorbar clamped).


In [ ]:

trainer.fit(tft, train_dataloader, val_dataloader)
trainer.save_checkpoint(os.path.join(ARTIFACT_DIR, "tft_txn_value_fixed.ckpt"))

Sanity Checking:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\user\Documents\imp_1\tftenv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:224: PossibleUserWarning: The dataloader, val_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 24 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


c:\Users\user\Documents\imp_1\tftenv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 24 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Epoch 0:   5%|▍         | 24/523 [00:10<03:43,  2.23it/s, loss=4.74e+03, v_num=0]

In [ ]:
# -----------------------
# 8. PREDICT & METRICS (correct evaluation over val set)
# -----------------------
# Get raw predictions and the input batch (return_x=True) to extract true decoder targets
raw_predictions, x = tft.predict(val_dataloader, mode="raw", return_x=True)

# Also get point forecasts (median / mean) with mode='prediction'
point_preds = tft.predict(val_dataloader, mode="prediction")  # shape: (num_samples, prediction_length)


In [ ]:
# The object x contains 'decoder_target' (true targets for each sample)
# and is a dict where decoder_target shape = (batch_size, prediction_length)
true_array = x['decoder_target'].detach().cpu().numpy()
pred_array = point_preds  # numpy

In [ ]:
# compute MAE/RMSE across all prediction windows (flatten)
true_flat = true_array.flatten()
pred_flat = pred_array.flatten()
mae_all = mean_absolute_error(true_flat, pred_flat)
rmse_all = mean_squared_error(true_flat, pred_flat, squared=False)

print("TFT overall (validation): MAE =", round(mae_all,3), "RMSE =", round(rmse_all,3))

In [ ]:
# compute cluster-wise metrics (approx): map validation samples back to biller -> cluster label
# x['encoder_cat'] or x['group_id'] may hold biller index; easiest: extract 'group_ids' from x
group_id_array = x['group_ids']  # tensor/list of strings representing biller_id per sample
group_id_list = [g.decode() if isinstance(g, bytes) else g for g in group_id_array]

In [ ]:
# Build dataframe for individual sample metrics (each sample is a series with prediction_length)
rows = []
for i, gid in enumerate(group_id_list):
    true_seq = true_array[i]
    pred_seq = pred_array[i]
    rows.append({
        'biller_id': gid,
        'mae': mean_absolute_error(true_seq, pred_seq),
        'rmse': mean_squared_error(true_seq, pred_seq, squared=False)
    })
sample_metrics_df = pd.DataFrame(rows)
# join cluster label
sample_metrics_df = sample_metrics_df.merge(biller_aggs_all[['biller_id','cluster_label']], on='biller_id', how='left')
cluster_perf = sample_metrics_df.groupby('cluster_label')[['mae','rmse']].mean().reset_index()
print("\nCluster-wise TFT metrics (mean over validation samples):")
print(cluster_perf)

In [ ]:
# -----------------------
# 9. VISUALIZATIONS: per-sample and aggregate
# -----------------------
# 9.1 plot first prediction sample using TFT's helper
fig = tft.plot_prediction(x, raw_predictions, idx=0, add_loss_to_title=True)
plt.show()

In [ ]:
# 9.2 Plot distribution of per-sample MAE
plt.figure(figsize=(8,4)); plt.hist(sample_metrics_df['mae'], bins=50); plt.title("Per-sample MAE distribution"); plt.show()


In [ ]:
# 9.3 Plot cluster-wise MAE bar chart
plt.figure(figsize=(8,4)); plt.bar(cluster_perf['cluster_label'], cluster_perf['mae']); plt.title("Cluster-wise mean MAE"); plt.xticks(rotation=45); plt.show()


In [ ]:

# 9.4 Example: compare actual vs predicted for 3 random billers from validation
random_billers = sample_metrics_df['biller_id'].drop_duplicates().sample(min(3, sample_metrics_df['biller_id'].nunique()), random_state=RANDOM_STATE).tolist()
for bid in random_billers:
    # find the first validation sample entry for this biller (there may be many)
    idx = sample_metrics_df[sample_metrics_df['biller_id']==bid].index[0]
    true_seq = true_array[idx]
    pred_seq = pred_array[idx]
    dates_idx = x['decoder_time_idx'][idx].detach().cpu().numpy()  # relative time idx
    # convert to real dates
    base_date = df['txn_date'].min()
    pred_dates = [base_date + pd.Timedelta(int(d), unit='D') for d in dates_idx]
    plt.figure(figsize=(9,3))
    plt.plot(pred_dates, true_seq, label='Actual')
    plt.plot(pred_dates, pred_seq, label='Predicted')
    plt.title(f"Biller {bid} - sample forecast")
    plt.legend()
    plt.show()


In [ ]:
# -----------------------
# 10. SAVE ARTIFACTS
# -----------------------
tft_state = {'trainer': trainer, 'model': tft}
joblib.dump({'tft_checkpoint': os.path.join(ARTIFACT_DIR, "tft_txn_value_fixed.ckpt")}, os.path.join(ARTIFACT_DIR,'tft_run_info.joblib'))
print("Saved artifacts to", ARTIFACT_DIR)

# End of notebook